# Bonus Analytics — B3 Monte Carlo + B4 Markowitz
## Bluestock MF Analytics — Bonus Deliverables
**B3:** Monte Carlo simulation projecting NAV growth over 5 years with uncertainty bands
**B4:** Markowitz Efficient Frontier portfolio optimisation for 5 selected funds


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
ROOT      = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"
REPORTS   = ROOT / "reports"

nav  = pd.read_csv(PROCESSED / "02_nav_history_clean.csv")
nav["date"] = pd.to_datetime(nav["date"])
nav  = nav[nav["date"].dt.dayofweek < 5].sort_values(["amfi_code","date"])
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
nav  = nav.dropna(subset=["daily_return"])

perf = pd.read_csv(PROCESSED / "07_scheme_performance_clean.csv")
print(f"✔ NAV records: {len(nav):,} | Schemes: {nav['amfi_code'].nunique()}")


## Bonus B3 — Monte Carlo Simulation
**Method:** Simulate 500 random NAV paths over 5 years using historical return distribution.
Each path is generated as: `NAV_t = NAV_0 × Π(1 + r_i)` where `r_i ~ N(μ, σ²)`


In [ ]:
top5 = perf[perf["category"].isin(
    ["Large Cap","Mid Cap","Small Cap"])].nlargest(5,"aum_crore")["amfi_code"].tolist()

np.random.seed(42)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, code_ in enumerate(top5):
    grp = nav[nav["amfi_code"]==code_].sort_values("date")
    r   = grp["daily_return"].dropna()
    mu, sig = r.mean(), r.std()
    current_nav = grp["nav"].iloc[-1]
    name = perf[perf["amfi_code"]==code_]["scheme_name"].iloc[0].split("-")[0].strip()[:22]

    n_days, n_sims = 252*5, 500
    sims = np.zeros((n_sims, n_days))
    for i in range(n_sims):
        daily_r = np.random.normal(mu, sig, n_days)
        sims[i] = current_nav * np.cumprod(1 + daily_r)

    ax = axes[idx]
    for i in range(100):
        ax.plot(sims[i], alpha=0.05, color="#3498db", linewidth=0.5)

    p10 = np.percentile(sims, 10, axis=0)
    p50 = np.percentile(sims, 50, axis=0)
    p90 = np.percentile(sims, 90, axis=0)

    ax.fill_between(range(n_days), p10, p90, alpha=0.2, color="#3498db", label="10–90th pct")
    ax.plot(p50, color="#e74c3c", linewidth=2, label="Median")
    ax.plot(p10, color="#f39c12", linewidth=1.5, linestyle="--", label="10th pct")
    ax.plot(p90, color="#2ecc71", linewidth=1.5, linestyle="--", label="90th pct")
    ax.axhline(current_nav, color="black", linewidth=1, linestyle=":", alpha=0.7, label="Current NAV")
    ax.set_title(f"{name}\nCurrent: ₹{current_nav:.0f}", fontsize=10, fontweight="bold")
    ax.set_xlabel("Trading Days"); ax.set_ylabel("NAV (₹)")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

axes[5].axis("off")
stats_lines = ["Monte Carlo Summary (5yr, 500 sims)\n"]
for code_ in top5:
    grp = nav[nav["amfi_code"]==code_]
    r   = grp["daily_return"].dropna()
    nav0 = grp["nav"].iloc[-1]
    name = perf[perf["amfi_code"]==code_]["scheme_name"].iloc[0].split("-")[0].strip()[:20]
    sims2 = np.zeros((200, 252*5))
    for i in range(200):
        dr = np.random.normal(r.mean(), r.std(), 252*5)
        sims2[i] = nav0 * np.cumprod(1+dr)
    final = sims2[:,-1]
    prob2x = (final >= nav0*2).mean()*100
    stats_lines.append(f"{name}:\n  Prob 2x in 5yr: {prob2x:.0f}%\n  Median: ₹{np.median(final):.0f}\n")
axes[5].text(0.05,0.95,"\n".join(stats_lines), transform=axes[5].transAxes,
             fontsize=9, verticalalignment="top", fontfamily="monospace")

fig.suptitle("Monte Carlo NAV Projection — 5 Years, 500 Simulations", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(REPORTS / "bonus_b3_monte_carlo.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Monte Carlo chart saved")


## Bonus B4 — Markowitz Efficient Frontier
**Method:** Generate 5,000 random portfolio weight combinations. Plot risk-return space.
Identify: Max Sharpe portfolio and Minimum Variance portfolio.
**Formula:** Portfolio Return = w·μ | Portfolio Risk = √(wᵀΣw)


In [ ]:
top5_codes = perf[perf["category"].isin(
    ["Large Cap","Mid Cap","Small Cap","Flexi Cap"])].nlargest(5,"aum_crore")["amfi_code"].tolist()

pivot = nav[nav["amfi_code"].isin(top5_codes)].pivot(
    index="date", columns="amfi_code", values="daily_return").dropna()

mu_vec  = pivot.mean().values * 252
cov_mat = pivot.cov().values  * 252
names   = [perf[perf["amfi_code"]==c]["scheme_name"].iloc[0].split("-")[0].strip()[:15]
           for c in top5_codes]
n = len(top5_codes)

np.random.seed(42)
N = 5000
rets, risks, sharpes = np.zeros(N), np.zeros(N), np.zeros(N)
weights_all = np.zeros((N, n))

for i in range(N):
    w = np.random.dirichlet(np.ones(n))
    ret  = np.dot(w, mu_vec)
    risk = np.sqrt(np.dot(w.T, np.dot(cov_mat, w)))
    rets[i]  = ret; risks[i] = risk
    sharpes[i] = (ret - 0.065) / risk
    weights_all[i] = w

fig, ax = plt.subplots(figsize=(12, 8))
sc = ax.scatter(risks*100, rets*100, c=sharpes, cmap="RdYlGn", alpha=0.5, s=8)
plt.colorbar(sc, ax=ax, label="Sharpe Ratio")

max_idx = np.argmax(sharpes); min_idx = np.argmin(risks)
ax.scatter(risks[max_idx]*100, rets[max_idx]*100, color="red", s=250, marker="*",
           zorder=5, label=f"Max Sharpe ({sharpes[max_idx]:.2f})")
ax.scatter(risks[min_idx]*100, rets[min_idx]*100, color="blue", s=200, marker="^",
           zorder=5, label="Min Variance")

for i, (name, ret, risk) in enumerate(zip(names, mu_vec*100, np.sqrt(np.diag(cov_mat))*100)):
    ax.scatter(risk, ret, color="black", s=120, marker="D", zorder=6)
    ax.annotate(name, (risk, ret), fontsize=8, ha="left", va="bottom")

ax.set_title("Markowitz Efficient Frontier — Top 5 Equity Funds", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Portfolio Risk (Annualised Std Dev %)"); ax.set_ylabel("Portfolio Return (Annualised %)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

max_w = weights_all[max_idx]
wt = "Max Sharpe Weights:\n" + "\n".join([f"  {nm}: {w*100:.1f}%" for nm,w in zip(names,max_w)])
ax.text(0.02,0.98,wt, transform=ax.transAxes, fontsize=9, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.tight_layout()
fig.savefig(REPORTS / "bonus_b4_efficient_frontier.png", dpi=150, bbox_inches="tight")
plt.show()

# Save results
opt = pd.DataFrame({"fund":names, "amfi_code":top5_codes,
                     "max_sharpe_weight_pct":(max_w*100).round(2),
                     "min_var_weight_pct":(weights_all[min_idx]*100).round(2)})
opt.to_csv(PROCESSED / "bonus_b4_optimal_weights.csv", index=False)
print("✔ Efficient Frontier chart saved")
print("✔ optimal_weights.csv saved")
print(f"\nMax Sharpe Portfolio (Sharpe = {sharpes[max_idx]:.3f}):")
print(opt[["fund","max_sharpe_weight_pct"]].to_string(index=False))


## Key Insights

**B3 — Monte Carlo:**
- All 5 equity funds show positive median NAV growth over 5 years
- Small Cap funds have the widest uncertainty bands — highest upside AND downside
- Large Cap funds have tighter bands — more predictable outcomes
- Probability of doubling NAV in 5 years is highest for Small Cap funds

**B4 — Markowitz:**
- The Efficient Frontier shows the best achievable return for each risk level
- Maximum Sharpe portfolio is NOT the same as maximum return — it balances risk and return
- Individual fund points lie BELOW the frontier — diversification always improves risk-adjusted returns
- Combining Large Cap + Small Cap in optimal weights can achieve higher Sharpe than any single fund
